# Pocket-Aware Uni-Mol Fine-Tuning on LP-PDBBind

Fine-tunes **Uni-Mol v2** (84M params) with **pocket context** on LP-PDBBind
(~19K protein-ligand complexes) for binding affinity prediction (pKa).

**Key difference from v1**: Uses pocket atom types + 3D coordinates concatenated
with ligand atoms, giving the model protein structural context.

**Requirements:** Colab Pro with A100 GPU runtime.

**Data flow:**
1. Download LP-PDBBind CSV + PDBBind refined-set structures
2. `build_pocket_ligand_data.py` extracts pocket+ligand atoms/coords
3. Uni-Mol trains on the combined pocket+ligand representation

## 1. Install Dependencies

In [ ]:
# Install unimol_tools without downgrading Colab's native GPU PyTorch
!pip install -q --no-deps unimol_tools
!pip install -q rdkit pandas scikit-learn matplotlib

import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\n⚠️ WARNING: GPU not available! Select A100/T4 runtime.')

import logging
logging.getLogger('numba').setLevel(logging.WARNING)

## 2. Mount Google Drive, Clone Repo & Download Data

In [ ]:
from google.colab import drive
import os, urllib.request

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/agentic-vlm'

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/handeboyaci/agentic-vlm.git "$PROJECT_DIR"
else:
    print('Repo already cloned, pulling latest...')
    !cd "$PROJECT_DIR" && git pull

%cd "$PROJECT_DIR"

# Download LP-PDBBind CSV
os.makedirs('data/pdbbind_deepchem/raw', exist_ok=True)
csv_path = 'data/pdbbind_deepchem/raw/LP_PDBBind.csv'
if not os.path.exists(csv_path):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/THGLab/LP-PDBBind/master/dataset/LP_PDBBind.csv',
        csv_path
    )
    print('Downloaded LP_PDBBind.csv')
else:
    print('LP_PDBBind.csv already exists.')

## 3A. Option A: SMILES-Only Training (Ligand-Only)

This is the simpler approach — trains on SMILES only (no pocket context).
Skip to Section 3B for pocket-aware training.

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem

CSV_PATH = 'data/pdbbind_deepchem/raw/LP_PDBBind.csv'
df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} complexes')

# Apply LP-PDBBind cleanup
if 'CL1' in df.columns:
    df = df[df['CL1'] & ~df['covalent']]
    print(f'After CL1 + non-covalent filter: {len(df)} complexes')

df_clean = df[['smiles', 'value', 'new_split']].copy()
df_clean = df_clean.rename(columns={'value': 'TARGET'})
df_clean = df_clean.dropna(subset=['smiles', 'TARGET'])

valid_mask = df_clean['smiles'].apply(lambda s: Chem.MolFromSmiles(str(s)) is not None)
df_clean = df_clean[valid_mask].reset_index(drop=True)

train_df = df_clean[df_clean['new_split'] == 'train'][['smiles', 'TARGET']]
test_df = df_clean[df_clean['new_split'] == 'test'][['smiles', 'TARGET']]
train_df = train_df.rename(columns={'smiles': 'SMILES'})
test_df = test_df.rename(columns={'smiles': 'SMILES'})

os.makedirs('data', exist_ok=True)
train_df.to_csv('data/unimol_train.csv', index=False)
test_df.to_csv('data/unimol_test.csv', index=False)
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

## 3B. Option B: Pocket-Aware Training (Recommended)

**This is the recommended approach.** Download the PDBBind refined-set
structures and build pocket+ligand data using `build_pocket_ligand_data.py`.

**Prerequisites:**
- PDBBind refined-set must be downloaded to `data/refined-set/`
- Each folder has `{pdb_id}_pocket.pdb` and `{pdb_id}_ligand.sdf`

In [ ]:
import pickle

POCKET_DATA_PATH = 'data/unimol_pocket_ligand.pkl'

# Build pocket+ligand data if not already built
if not os.path.exists(POCKET_DATA_PATH):
    if os.path.isdir('data/refined-set'):
        print('Building pocket+ligand data from refined-set...')
        !python scripts/build_pocket_ligand_data.py --output {POCKET_DATA_PATH}
    else:
        print('⚠️ data/refined-set/ not found.')
        print('Download PDBBind refined-set and extract to data/refined-set/')
        print('Or skip to Section 3A for SMILES-only training.')
else:
    print(f'Pocket+ligand data already built at {POCKET_DATA_PATH}')

# Load and inspect
if os.path.exists(POCKET_DATA_PATH):
    with open(POCKET_DATA_PATH, 'rb') as f:
        pocket_data = pickle.load(f)
    print(f'\nLoaded pocket+ligand data:')
    print(f'  Complexes: {len(pocket_data["atoms"])}')
    print(f'  Splits: {dict(zip(*np.unique(pocket_data["split"], return_counts=True)))}')
    print(f'  Sample atoms count: {len(pocket_data["atoms"][0])}')
    print(f'  pKa range: {pocket_data["target"].min():.1f} - {pocket_data["target"].max():.1f}')

## 4. Fine-Tune Uni-Mol v2

Choose either Section 4A (SMILES-only) or 4B (pocket-aware).

### 4A. SMILES-Only Training

In [ ]:
from unimol_tools import MolTrain

trainer = MolTrain(
    task='regression',
    data_type='molecule',
    epochs=100,
    patience=15,
    batch_size=16,
    metrics='mse,pearsonr',
    model_name='unimolv2',
    model_size='84m',
    save_path='./unimol_binding_model',
    remove_hs=False,
    target_cols='TARGET',
    use_cuda=True,
    use_amp=True,
    learning_rate=1e-4,
)

trainer.fit(data='data/unimol_train.csv')

### 4B. Pocket-Aware Training (Recommended)

Uses the pocket+ligand pickle from `build_pocket_ligand_data.py`.
The model sees protein pocket atoms + coordinates alongside the ligand.

In [ ]:
import pickle
import numpy as np
from unimol_tools import MolTrain

POCKET_DATA_PATH = 'data/unimol_pocket_ligand.pkl'

with open(POCKET_DATA_PATH, 'rb') as f:
    pocket_data = pickle.load(f)

# Split into train/test using the dataset's native split
splits = np.array(pocket_data['split'])
train_mask = splits == 'train'
test_mask = splits == 'test'

train_data = {
    'atoms': [pocket_data['atoms'][i] for i in range(len(splits)) if train_mask[i]],
    'coordinates': [pocket_data['coordinates'][i] for i in range(len(splits)) if train_mask[i]],
    'target': pocket_data['target'][train_mask],
}

test_data = {
    'atoms': [pocket_data['atoms'][i] for i in range(len(splits)) if test_mask[i]],
    'coordinates': [pocket_data['coordinates'][i] for i in range(len(splits)) if test_mask[i]],
    'target': pocket_data['target'][test_mask],
}

print(f'Train: {len(train_data["atoms"])}, Test: {len(test_data["atoms"])}')

# Train pocket-aware Uni-Mol
trainer = MolTrain(
    task='regression',
    data_type='molecule',          # unimol_tools handles atom+coord dicts
    epochs=100,
    patience=15,
    batch_size=16,
    metrics='mse,pearsonr',
    model_name='unimolv2',
    model_size='84m',
    save_path='./unimol_pocket_model',  # Different model dir!
    remove_hs=False,
    target_cols='target',
    use_cuda=True,
    use_amp=True,
    learning_rate=1e-4,
)

# Fit with custom dict data (atoms + coordinates + target)
trainer.fit(data=train_data)

## 5. Evaluate on Test Set

In [ ]:
import matplotlib.pyplot as plt
from unimol_tools import MolPredict
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr

# Choose the model to evaluate
MODEL_DIR = './unimol_pocket_model'  # or './unimol_binding_model' for SMILES-only
predictor = MolPredict(load_model=MODEL_DIR)

# Evaluate: for pocket-aware, use the test_data dict
# For SMILES-only, use: predictions = predictor.predict(data='data/unimol_test.csv')
if os.path.exists(POCKET_DATA_PATH):
    test_input = {
        'atoms': test_data['atoms'],
        'coordinates': test_data['coordinates'],
    }
    predictions = predictor.predict(data=test_input)
    y_true = test_data['target']
else:
    predictions = predictor.predict(data='data/unimol_test.csv')
    y_true = test_df['TARGET'].values

y_pred = predictions.flatten()

rmse = mean_squared_error(y_true, y_pred, squared=False)
mae = np.mean(np.abs(y_true - y_pred))
r, _ = pearsonr(y_true, y_pred)
rho, _ = spearmanr(y_true, y_pred)

print('\n' + '='*40)
print(f'Uni-Mol Test Results ({"Pocket-Aware" if os.path.exists(POCKET_DATA_PATH) else "Ligand-Only"})')
print('='*40)
print(f'RMSE:          {rmse:.4f} pKa units')
print(f'MAE:           {mae:.4f} pKa units')
print(f'Pearson r:     {r:.4f}')
print(f'Spearman rho:  {rho:.4f}')
print('='*40)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_true, y_pred, alpha=0.3, s=10, c='steelblue')
lims = [min(y_true.min(), y_pred.min()) - 0.5,
        max(y_true.max(), y_pred.max()) + 0.5]
ax.plot(lims, lims, 'r--', lw=1.5, label='Ideal')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('True pKa', fontsize=12)
ax.set_ylabel('Predicted pKa', fontsize=12)
mode = 'Pocket-Aware' if os.path.exists(POCKET_DATA_PATH) else 'Ligand-Only'
ax.set_title(f'Uni-Mol v2 ({mode})\nRMSE={rmse:.3f}, r={r:.3f}', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('unimol_scatter.png', dpi=150)
plt.show()

## 6. Export Model for Pipeline Integration

Download the model directory and place it in your VLM project:
- SMILES-only: `models/unimol_binding_model/`
- Pocket-aware: `models/unimol_pocket_model/`

The `unimol_predictor.py` auto-selects pocket model when available.

In [ ]:
import json as _json

# Zip the trained model
model_name = 'unimol_pocket_model' if os.path.exists(POCKET_DATA_PATH) else 'unimol_binding_model'
!zip -r {model_name}.zip {model_name}/

train_log = {
    'model': 'Uni-Mol v2 (84M)',
    'dataset': 'LP-PDBBind',
    'pocket_aware': os.path.exists(POCKET_DATA_PATH),
    'test_rmse': float(rmse),
    'test_pearson_r': float(r),
    'test_spearman_r': float(rho),
}
with open(f'training_log_{model_name}.json', 'w') as f:
    _json.dump(train_log, f, indent=2)

print(f'\n✅ Training complete!')
print(f'Download {model_name}.zip and place in models/{model_name}/')
print(f'Pipeline auto-selects pocket model: --scoring unimol')